# 🤖 AutoRig3D — UniRig AI 자동 리깅

GLB 3D 모델을 업로드하면 UniRig(SIGGRAPH 2025)가 자동으로 스켈레톤 + 스킨닝을 생성합니다.

**사용법:**
1. 런타임 → GPU로 변경 (T4 이상)
2. 셀을 순서대로 실행
3. GLB 파일 업로드
4. 리깅된 GLB 다운로드

## Step 0: GPU 확인

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB" if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "GPU가 필요합니다! 런타임 → 런타임 유형 변경 → T4 GPU 선택"

## Step 1: UniRig 설치 (최초 1회, ~3분)

In [ ]:
import os, subprocess

if not os.path.exists('/content/UniRig'):
    !git clone https://github.com/VAST-AI-Research/UniRig /content/UniRig
    %cd /content/UniRig

    # 1. bpy (Blender Python) — Colab용 빌드 설치
    !pip install -q bpy==4.3.0 2>/dev/null || \
     pip install -q bpy 2>/dev/null || \
     echo "⚠ bpy 설치 실패 — 아래에서 대안 처리"

    # 2. 핵심 의존성 먼저 설치
    !pip install -q lightning pytorch_lightning transformers einops omegaconf \
        python-box addict timm fast-simplification trimesh open3d pyrender \
        huggingface_hub wandb

    # 3. 나머지 requirements (bpy 제외)
    !sed -i '/^bpy/d' requirements.txt
    !pip install -q -r requirements.txt 2>&1 | tail -5

    # 4. spconv — CUDA 버전 자동 감지
    import torch
    cuda_ver = torch.version.cuda
    cuda_short = cuda_ver.replace('.', '')[:3]
    print(f"CUDA: {cuda_ver}")

    installed = False
    for ver in [cuda_short, cuda_short[:-1] + '1', cuda_short[:-1] + '0', '120']:
        r = subprocess.run(['pip', 'install', '-q', f'spconv-cu{ver}'],
                          capture_output=True, text=True)
        if r.returncode == 0:
            print(f"✅ spconv-cu{ver} 설치 완료")
            installed = True
            break
    if not installed:
        print("⚠ spconv 설치 실패")

    # 5. torch_scatter, torch_cluster
    torch_ver = torch.__version__.split('+')[0]
    !pip install -q torch_scatter torch_cluster \
        -f https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_short}.html 2>/dev/null || true

    # 6. bpy 설치 확인
    try:
        import bpy
        print("✅ bpy 사용 가능")
    except ImportError:
        print("⚠ bpy 미설치 — extract.py를 우회합니다")
        # extract.py의 bpy import를 우회하는 패치
        extract_path = '/content/UniRig/src/data/extract.py'
        if os.path.exists(extract_path):
            with open(extract_path, 'r') as f:
                content = f.read()
            content = content.replace('import bpy, os', 'import os  # bpy removed for Colab')
            content = content.replace('import bpy', '# import bpy  # removed for Colab')
            with open(extract_path, 'w') as f:
                f.write(content)
            print("  → extract.py 패치 완료")

    print("\n✅ UniRig 설치 완료!")
else:
    %cd /content/UniRig
    print("✅ UniRig 이미 설치됨")

# 설치 확인
print("\n--- 의존성 확인 ---")
for pkg in ['torch', 'lightning', 'transformers', 'trimesh', 'open3d']:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg}")

## Step 2: GLB 파일 업로드

In [ ]:
from google.colab import files
import shutil

# 업로드
print("GLB 파일을 업로드하세요...")
uploaded = files.upload()

# 입력 파일 저장
input_filename = list(uploaded.keys())[0]
input_path = f"/content/input/{input_filename}"
os.makedirs('/content/input', exist_ok=True)
with open(input_path, 'wb') as f:
    f.write(uploaded[input_filename])

print(f"✅ 업로드 완료: {input_filename} ({len(uploaded[input_filename]) / 1024:.0f}KB)")

## Step 3: 스켈레톤 생성 (~1-2분)

In [ ]:
os.makedirs('/content/output', exist_ok=True)
skeleton_path = '/content/output/skeleton.fbx'

print("🦴 스켈레톤 생성 중...")
!bash launch/inference/generate_skeleton.sh \
    --input "{input_path}" \
    --output "{skeleton_path}"

if os.path.exists(skeleton_path):
    print(f"✅ 스켈레톤 생성 완료: {os.path.getsize(skeleton_path) / 1024:.0f}KB")
else:
    print("❌ 스켈레톤 생성 실패")

## Step 4: 스킨닝 (웨이트 페인팅, ~1-2분)

In [ ]:
skin_path = '/content/output/skin.fbx'

print("🎨 스킨닝 생성 중...")
!bash launch/inference/generate_skin.sh \
    --input "{skeleton_path}" \
    --output "{skin_path}"

if os.path.exists(skin_path):
    print(f"✅ 스킨닝 완료: {os.path.getsize(skin_path) / 1024:.0f}KB")
else:
    print("❌ 스킨닝 실패")

## Step 5: 머지 (원본 메쉬 + 리깅 결합)

In [ ]:
rigged_path = '/content/output/rigged_model.glb'

print("🔗 머지 중...")
!bash launch/inference/merge.sh \
    --source "{skin_path}" \
    --target "{input_path}" \
    --output "{rigged_path}"

if os.path.exists(rigged_path):
    print(f"✅ 리깅 완료: {os.path.getsize(rigged_path) / 1024:.0f}KB")
else:
    print("❌ 머지 실패 — 스킨 FBX를 직접 다운로드합니다")
    rigged_path = skin_path

## Step 6: 결과 다운로드

In [ ]:
from google.colab import files

if os.path.exists(rigged_path):
    print(f"📥 다운로드: {rigged_path}")
    files.download(rigged_path)
else:
    print("❌ 리깅된 파일이 없습니다")

## (선택) Google Drive에 저장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/AutoRig3D'
os.makedirs(save_dir, exist_ok=True)

import shutil
output_name = input_filename.replace('.glb', '_rigged.glb')
shutil.copy2(rigged_path, f"{save_dir}/{output_name}")
print(f"✅ Google Drive 저장: {save_dir}/{output_name}")